In [2]:
!pip install -q transformers datasets peft accelerate beautifulsoup4 faiss-cpu sentence-transformers huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 43.6 MB/s eta 0:00:00


In [3]:
# Upgrade torchao to a compatible version
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 56.7 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [ ]:
import os

os.environ["HF_TOKEN"] = "hf_token"

# Hugging Face username
HF_USERNAME = "hf_username"

print("Done!")

Done!


In [ ]:
from google.colab import files
from bs4 import BeautifulSoup

# Upload files
html_files = ["EStG.html", "KStG.html", "UStG.html"]
uploaded = {f: open(f, "rb").read() for f in html_files}

# Extract plain text from each HTML file
documents = {}
for filename, content in uploaded.items():
    soup = BeautifulSoup(content, "html.parser")
    text = soup.get_text(separator="\n")
    # Clean up excessive blank lines
    text = "\n".join(line for line in text.splitlines() if line.strip())
    documents[filename] = text
    print(f"Extracted {len(text)} characters from {filename}")

print(f"\nLoaded {len(documents)} documents total")

Extracted 1634980 characters from EStG.html
Extracted 369672 characters from KStG.html
Extracted 513676 characters from UStG.html

Loaded 3 documents total


In [6]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

def split_into_chunks(text, chunk_size=500, overlap=50):
    """
    Splits a long text into smaller overlapping chunks.
    Smaller chunks so we can find more precise passages when searching.
    """
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

# Split all documents into chunks
all_chunks = []
for filename, text in documents.items():
    chunks = split_into_chunks(text)
    all_chunks.extend(chunks)
    print(f"{filename}: {len(chunks)} chunks")

print(f"\nTotal chunks: {len(all_chunks)}")

# Load a sentence embedding model
# This converts text into numbers so we can search by meaning
print("\nLoading embedding model...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Convert all chunks into embeddings
print("Embedding all chunks...")
chunk_embeddings = embedder.encode(all_chunks, show_progress_bar=True)

# Build a FAISS index for fast similarity search
dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(chunk_embeddings))

print(f"\nSearch index ready with {index.ntotal} chunks!")

EStG.html: 3634 chunks
KStG.html: 822 chunks
UStG.html: 1142 chunks

Total chunks: 5598

Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding all chunks...


Batches:   0%|          | 0/175 [00:00<?, ?it/s]


Search index ready with 5598 chunks!


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import login
import torch

# Logging into Hugging Face to access model
login(token=os.environ["HF_TOKEN"])

# Load model
MODEL_NAME = f"{HF_USERNAME}/finetuned-austrian-tax-gemma"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,  # 16-bit precision to save memory
    device_map="auto"           # Automatically use the GPU
)

print("Model loaded!")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Loading tokenizer...


tokenizer_config.json:   0%|          | 0.00/489 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/34.3M [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/591 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/3.70M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/72 [00:00<?, ?it/s]

Model loaded!


In [8]:
def rag_answer(question, top_k=3):
    """
    Given a question:
    1. Searches the tax law documents for the most relevant passages
    2. Passes those passages to the local Gemma model as context
    3. Returns the model's answer based on that context
    No API needed — everything runs locally!
    """
    # Step 1: Embed the question so we can search by meaning
    question_embedding = embedder.encode([question])

    # Step 2: Search the index for the most similar chunks
    distances, indices = index.search(np.array(question_embedding), top_k)

    # Step 3: Retrieve the actual text of the most relevant chunks
    relevant_chunks = [all_chunks[i] for i in indices[0]]
    context = "\n\n---\n\n".join(relevant_chunks)

    # Step 4: Build the prompt with context and question
    prompt = f"""### Question:
Use the following excerpts from Austrian tax law to answer the question concisely.

Context:
{context}

Question:
{question}

### Answer:
"""

    # Step 5: Tokenize and generate answer locally
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    # Extract only the answer part
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = full_output.split("### Answer:")[-1].strip()
    return answer

print("RAG function ready!")

RAG function ready!


In [9]:
import pandas as pd
from google.colab import files


# Load the dataset
df = pd.read_csv("/content/dataset_clean.csv")
print(f"Dataset loaded: {len(df)} rows")

# Loop through all questions in batches
BATCH_SIZE = 8
prompts = []
for _, row in df.iterrows():
    question = row["prompt"]
    # Embed the question and get relevant context
    question_embedding = embedder.encode([question])
    distances, indices = index.search(np.array(question_embedding), 3)
    relevant_chunks = [all_chunks[i] for i in indices[0]]
    context = "\n\n---\n\n".join(relevant_chunks)
    prompt = f"### Question:\nUse the following excerpts from Austrian tax law to answer the question concisely.\n\nContext:\n{context}\n\nQuestion:\n{question}\n\n### Answer:\n"
    prompts.append(prompt)

ids = [row["id"] for _, row in df.iterrows()]
results = []

print(f"\nStarting batched RAG inference on {len(prompts)} questions...\n")

for i in range(0, len(prompts), BATCH_SIZE):
    batch_prompts = prompts[i:i+BATCH_SIZE]
    batch_ids = ids[i:i+BATCH_SIZE]

    print(f"[{i+BATCH_SIZE}/{len(prompts)}] Processing batch...")

    inputs = tokenizer(
        batch_prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    for j, output in enumerate(outputs):
        full_output = tokenizer.decode(output, skip_special_tokens=True)
        answer = full_output.split("### Answer:")[-1].strip()
        results.append({
            "id": batch_ids[j],
            "answer": answer
        })

# Save output CSV
output_df = pd.DataFrame(results)
output_df.to_csv("submission_step3_rag_local.csv", index=False)

print(f"\nDone! Saved {len(output_df)} answers to 'submission_step3_rag_local.csv'")
print("\nOutput preview:")
print(output_df.head(3))

Dataset loaded: 643 rows

Starting batched RAG inference on 643 questions...

[8/643] Processing batch...
[16/643] Processing batch...
[24/643] Processing batch...
[32/643] Processing batch...
[40/643] Processing batch...
[48/643] Processing batch...
[56/643] Processing batch...
[64/643] Processing batch...
[72/643] Processing batch...
[80/643] Processing batch...
[88/643] Processing batch...
[96/643] Processing batch...
[104/643] Processing batch...
[112/643] Processing batch...
[120/643] Processing batch...
[128/643] Processing batch...
[136/643] Processing batch...
[144/643] Processing batch...
[152/643] Processing batch...
[160/643] Processing batch...
[168/643] Processing batch...
[176/643] Processing batch...
[184/643] Processing batch...
[192/643] Processing batch...
[200/643] Processing batch...
[208/643] Processing batch...
[216/643] Processing batch...
[224/643] Processing batch...
[232/643] Processing batch...
[240/643] Processing batch...
[248/643] Processing batch...
[256/